# ClusterAlgebras.jl - Interactive Tutorial

This notebook walks through all current functionality of `ClusterAlgebras.jl`.

**How to run:** Start Jupyter with the project environment active:
```sh
julia --project=. -e 'using IJulia; notebook(dir=".")'
```
Then open `examples/tutorial.ipynb`. If IJulia is not yet installed, run
```julia
using Pkg; Pkg.add("IJulia")
```
once in your *base* Julia environment (not inside `--project=.`).

---

### What is a cluster algebra?

A **cluster algebra** is a commutative ring generated by an (a priori infinite) collection of
rational functions called *cluster variables*, organised into overlapping sets of fixed size
called *clusters*. A cluster together with a quiver encoding the exchange relations between
its variables is called a **seed**. Clusters are connected by the operation of **mutation**.

The key result (Fomin–Zelevinsky, 2002): despite the exchange relation involving division,
every cluster variable is a **Laurent polynomial** in the initial variables. That is the
*Laurent phenomenon*.

**Contents of this tutorial:**
1. Quivers: construction and access
2. DOT / Graphviz export
3. Quiver mutation
4. Seeds
5. Seed mutation and the Laurent phenomenon
6. Working with algebra elements
7. Root systems
8. Finite-type detection and Cartan type recognition
9. Denominator vectors
10. Conway–Coxeter frieze patterns
11. Error types
12. Exploration playground

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using ClusterAlgebras

---
## 1. Quivers

A `Quiver` is a directed weighted graph encoded as a **skew-symmetrizable** integer matrix `B`.
`B[i,j] > 0` means there are `B[i,j]` arrows from vertex `i` to vertex `j`.

Skew-symmetrizable means there exist positive integers `d[1], …, d[n]` (the *symmetrizers*)
such that `d[i]·B[i,j] = −d[j]·B[j,i]` for all mutable `i, j`.
For **simply-laced** quivers (A, D, E Dynkin types) `d = [1,…,1]` and `B` is skew-*symmetric*.

### 1a. From an exchange matrix

In [ ]:
# The simplest non-trivial example: A₂ quiver  1 → 2
B = [0 1; -1 0]
q_A2 = Quiver(B)

In [ ]:
# Access fields directly
println("B matrix    : ", q_A2.B)
println("n_mutable   : ", q_A2.n_mutable)
println("n_frozen    : ", q_A2.n_frozen)
println("symmetrizers: ", q_A2.d)
println("labels      : ", q_A2.labels)

In [ ]:
# A₃ quiver: 1 → 2 → 3
B_A3 = [0 1 0; -1 0 1; 0 -1 0]
q_A3 = Quiver(B_A3)

### 1b. From an edge list

Use 2-tuples `(src, dst)` or 3-tuples `(src, dst, multiplicity)`.
Opposite-direction edges cancel automatically.

In [ ]:
# A₃ via edge list
q_from_edges = Quiver([(1,2), (2,3)])
println(q_from_edges.B)

In [ ]:
# With multiplicities - a quiver with a double arrow 1 ⇒ 2
q_double = Quiver([(1,2,2)])
println(q_double.B)

### 1c. Named Dynkin types

All finite Dynkin types are supported: `A`, `B`, `C`, `D`, `E` (rank 6, 7, 8), `F₄`, `G₂`.
Use a `Symbol` + rank integer, or a short string like `"A3"`.

In [ ]:
# Via Symbol + rank
q_D4 = Quiver(:D, 4)
q_D4

In [ ]:
# Via string (case-insensitive type letter)
q_E6 = Quiver("E6")
println("E₆ exchange matrix:")
display(q_E6.B)

In [ ]:
# Non-simply-laced types carry non-trivial symmetrizers d
q_B3 = Quiver(:B, 3)
println("B₃  B = ", q_B3.B, "  d = ", q_B3.d)

q_C3 = Quiver(:C, 3)
println("C₃  B = ", q_C3.B, "  d = ", q_C3.d)

q_F4 = Quiver(:F, 4)
println("F₄  B = ", q_F4.B, "  d = ", q_F4.d)

q_G2 = Quiver(:G, 2)
println("G₂  B = ", q_G2.B, "  d = ", q_G2.d)

### 1d. Frozen vertices

Frozen (or *coefficient*) vertices cannot be mutated and act as boundary conditions.
Pass `n_mutable` to specify how many of the first vertices are mutable.

In [ ]:
# 2 mutable vertices, 1 frozen vertex (vertex 3)
B_fr = [0  1  1;
       -1  0  1;
       -1 -1  0]
q_fr = Quiver(B_fr, 2)
q_fr   # show method marks frozen vertices

### 1e. Equality

In [ ]:
q1 = Quiver([0 1; -1 0])
q2 = Quiver(:A, 2)
q3 = Quiver("A2")
println("matrix == :A,2 ? ", q1 == q2)
println(":A,2  == \"A2\" ? ", q2 == q3)

---
## 2. DOT / Graphviz Export

`to_dot(q)` returns a Graphviz DOT string.  
Paste it at [graphviz.online](https://graphviz.online/) or pipe to the `dot` command-line tool.
Mutable vertices are circles; frozen vertices are boxes.

In [ ]:
println(to_dot(q_A2))

In [ ]:
# Weighted arrows get a label; frozen vertices use a box shape
println(to_dot(q_fr))

---
## 3. Quiver Mutation

Given a quiver with exchange matrix `B`, mutation at mutable vertex `k` produces a new
quiver with matrix `B'` given by the **Fomin–Zelevinsky rule**:

$$B'_{ij} = \begin{cases}
-B_{ij} & \text{if } i=k \text{ or } j=k \\
B_{ij} + \dfrac{|B_{ik}|B_{kj} + B_{ik}|B_{kj}|}{2} & \text{otherwise}
\end{cases}$$

Mutation is an **involution**: `μₖ ∘ μₖ = id`.

In [ ]:
# Single mutation
q_before = Quiver(:A, 3)
q_after  = mutate(q_before, 2)
println("A₃ before mutation at 2:")
println(q_before.B)
println("\nAfter mutation at 2:")
println(q_after.B)

In [ ]:
# Involutivity
println("μ₂ ∘ μ₂ = id? ", mutate(q_after, 2) == q_before)

In [ ]:
# Mutation sequence: pass a vector of vertex indices
q_seq = mutate(q_before, [1, 2, 1])
println("A₃ after μ₁μ₂μ₁:")
println(q_seq.B)

---
## 4. Seeds

A **seed** `(Q, 𝐱)` pairs a quiver `Q` with a *cluster* `𝐱 = (x₁, …, xₙ)` of
algebraically independent elements of the fraction field `Frac(ℤ[x₁,…,xₙ])`.
The ring is built automatically via **AbstractAlgebra.jl**.

In [ ]:
# Default variable names x1, x2, …
s = Seed(q_A2)
println("cluster : ", s.cluster)
println("ring    : ", s.ring)

In [ ]:
# Custom variable names
s_xy = Seed(q_A2, ["x", "y"])
println("cluster : ", s_xy.cluster)

In [ ]:
# Seeds with frozen variables - all n_mutable + n_frozen variables are in the cluster
s_fr = Seed(q_fr, ["x", "y", "c"])   # c is the coefficient / frozen variable
s_fr

---
## 5. Seed Mutation - the Exchange Relation

Mutating a seed at mutable vertex `k` replaces `xₖ` with `xₖ'`:

$$x_k' = \frac{\prod_{B_{ik}>0} x_i^{B_{ik}} + \prod_{B_{ik}<0} x_i^{-B_{ik}}}{x_k}$$

All other cluster variables are unchanged. The quiver mutates simultaneously.

In [ ]:
s0 = Seed(q_A2, ["x", "y"])

s1 = mutate(s0, 1)   # x → (1 + y) / x
println("After μ₁:  x₁' = ", s1.cluster[1])
println("           x₂' = ", s1.cluster[2])

s2 = mutate(s0, 2)   # y → (1 + x) / y
println("After μ₂:  x₁' = ", s2.cluster[1])
println("           x₂' = ", s2.cluster[2])

In [ ]:
# Involutivity: μ₁ ∘ μ₁ = id
s1_back = mutate(s1, 1)
println("μ₁∘μ₁ restores x? ", s1_back.cluster[1] == s0.cluster[1])
println("μ₁∘μ₁ restores y? ", s1_back.cluster[2] == s0.cluster[2])

### The Laurent Phenomenon - A₂ (5 cluster variables)

The A₂ cluster algebra is of **finite type** with exactly **5 cluster variables**.
Alternating mutations `μ₁μ₂μ₁μ₂μ₁` cycle through all of them and return to the start.

In [ ]:
s0 = Seed(Quiver(:A, 2), ["x", "y"])

orbit = [s0]
for k in [1, 2, 1, 2, 1]
    push!(orbit, mutate(orbit[end], k))
end

println("All 5 cluster variables of the A₂ cluster algebra:")
seen = Set{String}()
for s in orbit, v in s.cluster
    str = string(v)
    str in seen && continue
    push!(seen, str)
    println("  ", v)
end
println("\nReturned to initial cluster? ",
    Set(string.(orbit[end].cluster)) == Set(string.(orbit[1].cluster)))

### The Laurent Phenomenon - A₃ (9 cluster variables)

In [ ]:
s_A3 = Seed(Quiver(:A, 3), ["x", "y", "z"])

seen_vars = Set{String}()
for v in s_A3.cluster; push!(seen_vars, string(v)); end

current = s_A3
for _ in 1:20, k in [1, 2, 3, 2, 1, 3]
    current = mutate(current, k)
    for v in current.cluster
        push!(seen_vars, string(v))
    end
end

println("Distinct A₃ cluster variables found: ", length(seen_vars))
for v in sort(collect(seen_vars))
    println("  ", v)
end

---
## 6. Working with the Algebra Elements

Cluster variables live in the fraction field of a multivariate polynomial ring over `ℤ`.
You can do any arithmetic on them directly.

In [ ]:
s0 = Seed(Quiver(:A, 2), ["x", "y"])
s1 = mutate(s0, 1)

x, y    = s0.cluster
x_new   = s1.cluster[1]   # = (1 + y) / x

println("x_new         = ", x_new)
println("x · x_new     = ", x * x_new)      # should be 1 + y
println("x · x_new − 1 = ", x * x_new - 1)  # should be y

In [ ]:
# Any AbstractAlgebra fraction-field arithmetic
gen_x, gen_y = s0.cluster
expr = (gen_x^2 + gen_y^2) // (gen_x * gen_y)
println("Custom expression: ", expr)

---
## 7. Root Systems

A surprising amount of cluster algebra theory is really root-system bookkeeping.
`RootSystem(type, n)` computes the positive roots (via BFS on the Cartan matrix),
the **Coxeter number** `h`, the **fundamental exponents** `e₁ ≤ … ≤ eₙ`, and
the **Weyl group order** `|W|`, all from classical Lie-theory tables.

In [ ]:
rs_A3 = RootSystem(:A, 3)
println("A₃ root system")
println("  Coxeter number h     = ", rs_A3.coxeter_number)    # 4
println("  Fundamental exponents = ", rs_A3.exponents)         # [1, 2, 3]
println("  Weyl group order |W|  = ", rs_A3.weyl_group_order)  # 24
println("  # positive roots      = ", length(rs_A3.positive_roots))  # 6

In [ ]:
# Positive roots in simple-root coordinates
println("Positive roots of A₃ (simple-root basis):")
for r in rs_A3.positive_roots
    println("  ", r)
end

In [ ]:
# Almost-positive roots: negative simple roots ∪ positive roots
# These are in bijection with cluster variables in finite type (Fomin–Zelevinsky II)
apr_A3 = almost_positive_roots(rs_A3)
n, h   = rs_A3.n, rs_A3.coxeter_number

println("Almost-positive roots of A₃ (−simples first, then positives):")
for r in apr_A3
    println("  ", r)
end
println()
println("Total: ", length(apr_A3), " (formula n(h+2)/2 = ", n*(h+2)÷2, ")")

In [ ]:
# Weyl group order table for all finite types
for (t, n) in [(:A,4), (:B,3), (:C,3), (:D,4), (:E,6), (:F,4), (:G,2)]
    rs = RootSystem(t, n)
    println("$t$n: h = $(rs.coxeter_number), |W| = $(rs.weyl_group_order), ",
            "# pos roots = $(length(rs.positive_roots))")
end

In [ ]:
# Cartan companion of a quiver: diagonal 2, off-diagonal −|B[i,j]|
A = cartan_companion(Quiver(:G, 2))
println("Cartan companion of G₂:")
display(A)

---
## 8. Finite-Type Detection and Cartan Type Recognition

**Finite type** means the cluster algebra has only finitely many cluster variables,
which happens iff the symmetrized Cartan companion `D·A` is **positive definite**.
This is checked *exactly* via Sylvester's criterion - no floating point.

> **Warning:** *finite type* (finitely many cluster variables) is strictly stronger than
> *mutation-finite* (finitely many quivers in the mutation class).
> The Markov quiver below is mutation-finite but **not** finite type.

In [ ]:
# All named Dynkin quivers are finite type
for (t, n) in [(:A,3), (:B,3), (:C,3), (:D,4), (:E,6), (:F,4), (:G,2)]
    q = Quiver(t, n)
    println("$t$n finite type? ", is_finite_type(q))
end

In [ ]:
# Affine types are positive semidefinite - finitely many quivers but infinitely many cluster variables
# Ã₂: 3 vertices in a 3-cycle (the smallest affine type)
B_affine_A2 = [0 1 -1; -1 0 1; 1 -1 0]
q_affine = Quiver(B_affine_A2)
println("Ã₂ is finite type? ", is_finite_type(q_affine))
println("Ã₂ is affine type? ", is_affine_type(q_affine))

In [ ]:
# The Markov quiver: three vertices, all double arrows in a cycle
# It is mutation-finite and indefinite - neither finite nor affine type
B_markov = [0 2 -2; -2 0 2; 2 -2 0]
q_markov = Quiver(B_markov)
println("Markov quiver is finite type? ", is_finite_type(q_markov))   # false
println("Markov quiver is affine type? ", is_affine_type(q_markov))   # false
println("(It is mutation-finite but generates infinitely many cluster variables)")

In [ ]:
# cartan_type identifies the Dynkin type of any finite-type acyclic quiver
for (t, n) in [(:A,4), (:D,5), (:E,8), (:F,4), (:G,2)]
    q = Quiver(t, n)
    println("cartan_type(Quiver(:$t,$n)) = ", cartan_type(q))
end

In [ ]:
# Type recognition works on any acyclic mutation-equivalent quiver, not just the standard one
# Here we mutate the A₃ quiver and check the result is still recognized as A₃
q_A3_mut = mutate(Quiver(:A, 3), 2)   # mutated quiver has different B matrix
println("Original A₃ B:  ", Quiver(:A, 3).B)
println("Mutated  A₃ B:  ", q_A3_mut.B)
println("Type of mutated quiver: ", cartan_type(q_A3_mut))

---
## 9. Denominator Vectors

Every cluster variable `x` has a **denominator vector** `d(x) ∈ ℤⁿ`:
the exponent vector of the denominator monomial when `x` is written in lowest terms
in the initial cluster `(x₁,…,xₙ)`. For the initial variables, `d(xᵢ) = −eᵢ`
(Fomin–Zelevinsky convention).

In **finite type**, the denominator vectors of all cluster variables are exactly
the **almost-positive roots** of the corresponding root system: the negative simple
roots (initial variables) together with all positive roots (non-initial variables).

In [ ]:
# A₂: denominator vectors of the initial cluster
s0 = Seed(Quiver(:A, 2), ["x", "y"])
println("d-vector of x (initial var 1): ", denominator_vector(s0, 1))  # [-1,  0]
println("d-vector of y (initial var 2): ", denominator_vector(s0, 2))  # [ 0, -1]

In [ ]:
# After mutation at vertex 1: x → (1+y)/x, denominator is x¹ → d = [1, 0]
s1 = mutate(s0, 1)
println("After μ₁:")
println("  cluster var 1 = ", s1.cluster[1])  # (1+y)/x
println("  d-vector      = ", denominator_vector(s1, 1))  # [1, 0]

In [ ]:
# Trace all 5 cluster variables of A₂ and their d-vectors
s0 = Seed(Quiver(:A, 2), ["x", "y"])
orbit = [s0]
for k in [1, 2, 1, 2, 1]
    push!(orbit, mutate(orbit[end], k))
end

dvecs = Set{Tuple{String,Vector{Int}}}()
for s in orbit, k in 1:s.quiver.n_mutable
    v = s.cluster[k]
    push!(dvecs, (string(v), denominator_vector(s, k)))
end

println("A₂ cluster variable   →   d-vector")
for (var, dv) in sort(collect(dvecs); by=first)
    println("  ", rpad(var, 24), dv)
end

In [ ]:
# The multiset of d-vectors equals the almost-positive roots of A₂
rs_A2  = RootSystem(:A, 2)
apr_A2 = almost_positive_roots(rs_A2)

dvecs_only = [dv for (_, dv) in dvecs]
println("d-vectors collected   : ", sort(dvecs_only))
println("almost-positive roots : ", sort(apr_A2))
println("Sets match?           : ", sort(dvecs_only) == sort(apr_A2))

---
## 10. Conway–Coxeter Frieze Patterns

A **Conway–Coxeter SL₂ frieze** of order `n` is a strip of positive integers with:
- A border row of `0`s on top and bottom, bordered by rows of `1`s
- The **unimodular diamond rule**: every 2×2 adjacent sub-matrix has determinant 1

Friezes are the type-`A_{n-3}` cluster algebra specialized at initial variables = 1.
They connect cluster mutation, triangulations of polygons, and continued fractions.

`frieze(n)` uses the **fan triangulation** (all diagonals from vertex 1),
giving quiddity `(n−2, 1, 2, 2, …, 2, 1)`.

In [ ]:
# n = 5: the pentagon frieze (A₂, width 2)
f5 = frieze(5)
println(f5)

In [ ]:
# n = 6: the hexagon frieze (A₃, width 3)
f6 = frieze(6)
println(f6)

In [ ]:
# n = 7: the heptagon frieze (A₄, width 4)
f7 = frieze(7)
println(f7)

In [ ]:
# Verify the unimodular diamond rule at every position
using AbstractAlgebra   # brings is_valid into scope
println("Pentagon frieze valid?  ", is_valid(f5))
println("Hexagon  frieze valid?  ", is_valid(f6))
println("Heptagon frieze valid?  ", is_valid(f7))

In [ ]:
# All interior entries are positive integers
n = f6.n
interior = f6.entries[3:n-1, :]   # rows 3..n-1 in Julia = interior rows
println("Interior rows of hexagon frieze:")
display(interior)
println("All positive? ", all(>(0), interior))

In [ ]:
# The quiddity sequence encodes the triangulation
println("Pentagon quiddity: ", f5.quiddity)   # fan triangulation of 5-gon
println("Hexagon  quiddity: ", f6.quiddity)
println("Width (= n − 3) for n=5: ", f5.n - 3)
println("Width (= n − 3) for n=6: ", f6.n - 3)

In [ ]:
# Custom quiddity: a 3-fold-symmetric triangulation of the hexagon
# Triangulation (1,3,5) - the alternating inner triangle - gives quiddity (3,1,3,1,3,1)
f_sym = frieze([3, 1, 3, 1, 3, 1])
println("Symmetric hexagon frieze:")
println(f_sym)
println("Valid? ", is_valid(f_sym))

In [ ]:
# Glide-reflection symmetry: row 3 of the hexagon frieze is a cyclic shift of row 5
F  = f6.entries
r3 = F[3, :]
r5 = F[5, :]
shift = findfirst(s -> circshift(r5, s) == r3, 0:5)
println("Row 3 (quiddity) : ", r3)
println("Row 5 (interior) : ", r5)
println("Glide shift      : ", shift - 1, " columns")

### Connection: frieze entries = cluster variables at 1

The entries of the A₂ frieze (n = 5) are exactly the 5 cluster variables of A₂
specialized at `x = y = 1`. We can verify this directly.

In [ ]:
# Collect the 5 A₂ cluster variables symbolically, then evaluate at x=y=1
s0 = Seed(Quiver(:A, 2), ["x", "y"])
orbit = [s0]
for k in [1, 2, 1, 2, 1]
    push!(orbit, mutate(orbit[end], k))
end

R, (x, y) = s0.cluster[1].parent.base_ring, gens(s0.cluster[1].parent.base_ring)

seen_strs = Set{String}()
vals = Int[]
for s in orbit, v in s.cluster
    str = string(v)
    str in seen_strs && continue
    push!(seen_strs, str)
    # Evaluate at x=1, y=1: substitute the fraction field element
    num_val = evaluate(numerator(v), [1, 1])
    den_val = evaluate(denominator(v), [1, 1])
    push!(vals, num_val ÷ den_val)
    println("  ", rpad(str, 24), " → ", num_val ÷ den_val)
end
println("\nFrieze interior of n=5 (up to cyclic order): ", sort(vals))
println("Frieze row 3 of f5                          : ", sort(f5.entries[3,:]))

---
## 11. Error Types

The package defines a hierarchy of typed errors under `ClusterAlgebraError`, each with a
message naming the specific problem.

In [ ]:
# NotSkewSymmetrizable - shows the offending (i,j) entries
try
    Quiver([0 2; -3 0], 2, [1, 1])  # d=[1,1] but 1·2 ≠ −1·(−3)
catch e
    println(typeof(e), " <: ClusterAlgebraError? ", e isa ClusterAlgebraError)
    println(e)
end

In [ ]:
# FrozenVertexMutation - names the vertex and why it cannot be mutated
try
    mutate(Quiver([0 1 0; -1 0 1; 0 -1 0], 2), 3)
catch e
    println(typeof(e))
    println(e)
end

In [ ]:
# InvalidVertex - out-of-range
try
    mutate(Quiver(:A, 3), 99)
catch e
    println(typeof(e))
    println(e)
end

In [ ]:
# InvalidArgument - various: unknown type, wrong rank, bad frieze input
try
    Quiver(:E, 5)       # E type only defined for n ∈ {6,7,8}
catch e; println(e); end

try
    Quiver("Z5")        # unknown Dynkin letter
catch e; println(e); end

try
    frieze(3)           # need n ≥ 4
catch e; println(e); end

try
    frieze([1, -1, 2, 3])   # non-positive quiddity entry
catch e; println(e); end

---
## 12. Exploration Playground

Some larger examples to experiment with.

In [ ]:
# D₄ mutation path: mutate and reverse to check involutivity
q0   = Quiver(:D, 4)
path = [1, 2, 3, 4, 2, 1, 3]
println("D₄ roundtrip: ", mutate(mutate(q0, path), reverse(path)) == q0)

In [ ]:
# G₂ root system - the exceptional rank-2 type with a triple bond
rs_G2 = RootSystem(:G, 2)
println("G₂: h = ", rs_G2.coxeter_number,
        ", exponents = ", rs_G2.exponents,
        ", |W| = ", rs_G2.weyl_group_order)
println("Positive roots:")
for r in rs_G2.positive_roots
    println("  ", r)
end

In [ ]:
# Denominator vectors for A₃ - should match almost-positive roots
s0    = Seed(Quiver(:A, 3), ["x", "y", "z"])
rs_A3 = RootSystem(:A, 3)
apr   = Set(almost_positive_roots(rs_A3))

dvecs_A3 = Set{Vector{Int}}()
current = s0
for _ in 1:30, k in [1, 2, 3, 2, 1, 3]
    current = mutate(current, k)
    for j in 1:3
        push!(dvecs_A3, denominator_vector(current, j))
    end
end
for j in 1:3; push!(dvecs_A3, denominator_vector(s0, j)); end

println("A₃ d-vectors found:       ", length(dvecs_A3))
println("A₃ almost-positive roots: ", length(apr))
println("Sets agree?               ", dvecs_A3 == apr)

In [ ]:
# Larger friezes
for n in 4:9
    f  = frieze(n)
    ok = is_valid(f)
    mx = maximum(f.entries)
    println("n = $n (width $(n-3)): max entry = $mx, valid = $ok")
end

In [ ]:
# The Markov quiver generates the famous Markov triples x²+y²+z²=3xyz
# starting from (1,1,1), each mutation gives the next Markov number
s_mk = Seed(Quiver([0 2 -2; -2 0 2; 2 -2 0]), ["a", "b", "c"])
println("Markov mutations (variables specialize to Markov triples at a=b=c=1):")
current = s_mk
for k in [1, 2, 3, 1]
    current = mutate(current, k)
    R = current.cluster[1].parent.base_ring
    vals = [Int(evaluate(numerator(v), [1,1,1]) ÷ evaluate(denominator(v), [1,1,1]))
            for v in current.cluster]
    println("  After μ$k: ", vals)
end

In [ ]:
# Build a quiver via edge list and check its type
q_custom = Quiver([(1,2), (2,3), (3,4), (4,2)])   # a 3-cycle with a tail
println(q_custom)
println("Finite type? ", is_finite_type(q_custom))
println("Affine type? ", is_affine_type(q_custom))